<a href="https://colab.research.google.com/github/eeeewyz/RAG/blob/main/5_vector%20database_Weaviate%20API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ungraded Lab - Introduction to Weaviate API


<div align="center">
  <img src="images/weaviate.png" alt="RAG Overview" width="10%">
</div>

Welcome to the ungraded lab about the Weaviate API! As you dive into the world of vector databases, you'll discover that there are several options available to help you build your RAG systems. In this course, you'll focus on the [Weaviate API](https://weaviate.io/).

This lab is designed to give you a hands-on introduction to Weaviate, so you'll be well-prepared for the upcoming assignment. You'll explore how Weaviate functions, what it can do, and how to make the most of its features. By the time you reach the assignment, you'll have the tools and knowledge you need to succeed.

Let's go!


# Table of Contents
- [ 1 - Introduction](#1)
  - [ 1.1 Loading the necessary libraries](#1-1)
  - [ 1.2 - The Weaviate Client](#1-2)
- [ 2 - Configuring the database](#2)
  - [ 2.1 Creating a Collection](#2-1)
  - [ 2.2 Configuring the Vectorizer](#2-2)
  - [ 2.3 The Properties](#2-3)
  - [ 2.4 Adding elements into a Collection](#2-4)
- [ 3 - Querying on a collection](#3)
  - [ 3.1 Filters](#3-1)
  - [ 3.2 Semantic Search](#3-2)
  - [ 3.3 BM25 search](#3-3)
  - [ 3.4 Hybrid Search](#3-4)
  - [ 3.5 Reranking](#3-5)


---
<h4 style="color:black; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>

JupyterLab provides an easy way for you to navigate through your assignment. It's located under the Table of Contents tab, found in the left panel, as shown in the picture below.

![TOC Location](images/toc.png)

---

<a id='1'></a>
## 1 - Introduction
---
<a id='1-1'></a>
### 1.1 Loading the necessary libraries
Run the cell below to load the necesary libraries for this assignment.

In [ ]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os

Flask 是一个 Python 的轻量级 Web 框架，主要用来把 Python 程序做成网页应用或 API。

In [ ]:
import flask_app
from utils import (
    suppress_subprocess_output,
    generate_with_single_input,
    print_object_properties
)

<a id='1-2'></a>
### 1.2 - The Weaviate Client

To start working with the Weaviate API in this environment, you need to start a `client`. In this course, you will use an *embedded client*, which is a way of using Weaviate within this application and not relying on a stand-alone instance of Weaviate running.

When you start `Embedded Weaviate` for the first time, it creates a data storage file at the location you specify in `persistence_data_path`. Even after you close your client and Embedded Weaviate shuts down, your data will still be saved there.

When creating your Weaviate client, you must pass an embedding model to perform vectorization. You can pass different models and Weaviate has different modules to help you call `OpenAI` models and others. Since OpenAI is a paid system, we will use a local model to perform the vectorization.

One example to load an OpenAI model is this call:


```Python
import weaviate
client = weaviate.connect_to_embedded(
    version="1.26.1",
    headers={
        "X-OpenAI-Api-Key": YOUR_OPENAI_API_KEY
    },
)
```

Let's load our client!

This function `suppress_subprocess_output()` is designed to suppress the Weaviate output logs, which can pollute the lab. These logs won't be explored in this lab, but feel free to remove it if you are curious about what the logs look like! The arguments it will be using are:

- `persistence_data_path`: The path where the client will look for (and create) the vector databases. Once you create it, it is stored there and persisted, i.e., it won't be deleted once you close the client!
- `environment_vabiables`: Necessary variables that we must pass to make the local embedding server to work.

### 作用

这段代码用于启动本地 Weaviate 向量数据库，并创建 `client` 客户端。

- 数据保存在 `./.collections`
- Flask 服务运行在 `127.0.0.1:5000`
- `text2vec-transformers` 负责把文本转换为向量
- `reranker-transformers` 负责对检索结果重新排序

整体流程：

用户查询 → Flask 生成向量 → Weaviate 检索 → Reranker 重新排序

# 隐藏 Weaviate 启动时产生的日志输出
with suppress_subprocess_output():

    # 启动并连接本地嵌入式 Weaviate 向量数据库
    client = weaviate.connect_to_embedded(

        # 指定数据库文件的保存位置，关闭后数据不会丢失
        persistence_data_path="./.collections",

        # 配置 Weaviate 运行时所需的环境变量
        environment_variables={

            # 允许使用基于 API 的模型模块
            "ENABLE_API_BASED_MODULES": "true",

            # 启用文本向量化模块和结果重排序模块
            "ENABLE_MODULES": "text2vec-transformers, reranker-transformers",

            # 指定 Embedding 模型服务地址
            # Weaviate 会调用该接口把文本转换成向量
            "TRANSFORMERS_INFERENCE_API": "http://127.0.0.1:5000/",

            # 指定 Reranker 模型服务地址
            # 用于对初步检索结果重新排序
            "RERANKER_INFERENCE_API": "http://127.0.0.1:5000/"
        }
    )

In [ ]:
with suppress_subprocess_output():
    client = weaviate.connect_to_embedded(
        persistence_data_path="./.collections",
        environment_variables={
            "ENABLE_API_BASED_MODULES": "true", # Enable API based modules
            "ENABLE_MODULES": 'text2vec-transformers, reranker-transformers', # We will be using a transformer model
            "TRANSFORMERS_INFERENCE_API":"http://127.0.0.1:5000/", # The endpoint the weaviate API will be using to vectorize
            "RERANKER_INFERENCE_API":"http://127.0.0.1:5000/" # The endpoint the weaviate API will be using to rerank
        }
    )

WeaviateStartUpError: Embedded DB did not start because processes are already listening on ports http:8079 and grpc:50050use weaviate.connect_to_local(port=8079, grpc_port=50050) to connect to the existing instance

With the defined `client`, your primary usage is creating a collection, adding elements to it and querying over it.

<a id='2'></a>
## 2 - Configuring the database

---

In this section, you will explore the central object in this lab and in this assignment: [the collection](https://weaviate.io/developers/weaviate/manage-data/collections) - this is the name Weaviate gives to a group of data objects which will be indexed for retrieval. Remember the workflow from the lectures:

<div align="center">
  <img src="images/workflow.png" alt="RAG Overview" width="60%">
</div>


<a id='2-1'></a>
### 2.1 Creating a Collection

To create a collection, there are some parameters that must be set. The most important for our purposes are:

- `name`: the collection name, this is the name that will be saved in memory and the name that you will need to load it.
- `vectorizer_config`: a list with vectorizer configurations. You can pass more than one vectorizer configuration, which means that in the same vector database, you can vectorize your datapoints with different embedding models. In your context, you will be using only one.

Let's load a database to illustrate this section.

In [ ]:
data = joblib.load("data.joblib")
print_object_properties(data[0])

In [ ]:
len(data)

The dataset is a set of places to visit, with some properties describing each location. The properties here are `place, state, description, best_season_to_visit, attractions, budget, user_ratings, last_updated`. When creating a collection, you must create one property for each key in this dictionary and add the expected datatype.

<a id='2-2'></a>
### 2.2 Configuring the Vectorizer

As mentioned before, you will use the `text2vec_transformers` embedding model to vectorize your data. To configure it, you must pass the corresponding Configure object. When configuring the vectorizer, you can pass a list of different vectorizers, so your collection can store several vectorizations for the same object. You can also choose to vectorize specific properties on specific vectorizers. In this course you will stick with one vectorizer. Not every property must be vectorized, it depends on the data and the information you want to retrieve.

In this case, let's use the following properties to be vectorized:

`place, state, description, best_season_to_visit, attractions, budget`

These properties will be appended to each other and then vectorized. When defining the property, you might choose to add the property name or not in the vectorization. Note that it would make sense to have the property name in budget, for example, as only the word "Moderate" would not provide enough information about what "Moderate" stands for.

In [ ]:
# 配置 Collection 中对象的向量生成方式
vectorizer_config = [
    Configure.NamedVectors.text2vec_transformers(

        # 给生成的向量命名为 "vector"
        # 后续查询或读取对象向量时，需要使用这个名称
        name="vector",

        # 指定哪些字段参与生成向量
        # Weaviate 会把这些字段的文本拼接起来，再发送给 Embedding 模型
        source_properties=[
            "place",
            "state",
            "description",
            "best_season_to_visit",
            "attractions",
            "budget"
        ],

        # 不把 Collection 的名称加入待向量化文本
        # 如果设为 True，Collection 名称会被拼接到文本最前面
        vectorize_collection_name=False,

        # 指定本地 Embedding 模型服务的地址
        # Weaviate 会调用这个 Flask API，将文本转换成向量
        inference_url="http://127.0.0.1:5000"
    )
]

<a id='2-3'></a>
### 2.3 The Properties

In a collection, the features of each data point are called Properties.

In [ ]:
# 如果名为 "example_collection" 的 collection 已经存在，
# 则先将其删除，避免后续重复创建时发生名称冲突。
if client.collections.exists("example_collection"):
    client.collections.delete("example_collection")

In [ ]:
# 如果 "example_collection" 不存在，则创建它
if not client.collections.exists("example_collection"):

    collection = client.collections.create(
        name="example_collection",

        # 使用之前定义好的向量化配置。
        # 指定哪些字段参与向量生成，以及调用哪个向量化模型/API。
        vectorizer_config=vectorizer_config,

        # 配置 reranker。
        # 初步检索出候选结果后，reranker 会再次计算相关性并重新排序。
        reranker_config=Configure.Reranker.transformers(),

        # 定义 collection 中每条数据所包含的字段
        properties=[
            Property(
                name="place",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            Property(
                name="state",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            Property(
                name="description",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            Property(
                name="best_season_to_visit",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            Property(
                name="attractions",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            Property(
                name="budget",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 数值类型字段，通常用于过滤、排序或统计，
            # 不会直接作为文本生成向量。
            Property(
                name="user_ratings",
                data_type=DataType.NUMBER
            ),

            # 日期类型字段，可用于按照更新时间进行过滤或排序。
            Property(
                name="last_updated",
                data_type=DataType.DATE
            ),
        ]
    )

# 如果 collection 已经存在，则不重复创建，直接获取现有 collection
else:
    collection = client.collections.get("example_collection")

Running it creates a collection and returns the collection. Printing it shows the collection configuration.

In [ ]:
print("Collection type:", type(collection))

In [ ]:
#1. Collection 创建后默认是空的
# 创建好 collection 只是定义了数据结构，还没有真正的数据。
len(collection)

In [ ]:
print(collection)

If you try to create a collection that already exists, an exception will be thrown:

In [ ]:
try:
    # 尝试创建一个名为 example_collection 的 collection
    collection = client.collections.create(
        name="example_collection",

        # 使用之前定义好的向量化配置
        # 决定使用哪个向量模型，以及哪些字段用于生成向量
        vectorizer_config=vectorizer_config,

        # 定义 collection 中每条数据包含的字段
        properties=[
            # 地点名称，文本类型
            # vectorize_property_name=True：
            # 生成向量时，字段名 "place" 也会和字段值一起参与向量化
            Property(
                name="place",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 州或地区名称
            Property(
                name="state",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 地点的详细介绍
            Property(
                name="description",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 最适合旅游的季节
            Property(
                name="best_season_to_visit",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 主要景点
            Property(
                name="attractions",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 预算信息
            Property(
                name="budget",
                vectorize_property_name=True,
                data_type=DataType.TEXT
            ),

            # 用户评分，数字类型
            # 可用于数值过滤和排序，例如筛选评分大于 4.5 的地点
            Property(
                name="user_ratings",
                data_type=DataType.NUMBER
            ),

            # 最后更新时间，日期类型
            # 可用于按时间过滤或排序
            Property(
                name="last_updated",
                data_type=DataType.DATE
            ),
        ]
    )

except Exception as e:
    # 如果创建过程中发生错误，就捕获异常并打印错误信息
    # 例如 collection 已经存在、配置错误或服务器连接失败
    print(e)

You can also retrieve all the collections saved:

.list_all().keys() 是两个方法连续调用：

对象.list_all().keys()

含义是：

list_all()：先获取全部数据，通常返回一个字典。
keys()：再取出这个字典中的所有键。

例如：

students = {
    "Tom": 90,
    "Alice": 85,
    "Bob": 78
}

students.keys()

In [ ]:
students = {
    "Tom": 90,
    "Alice": 85,
    "Bob": 78
}

students.keys()

In [ ]:
client.collections.list_all().keys()

The result of .list_all() is a dictionary with the collections names as keys and their properties.

<a id='2-4'></a>
### 2.4 Adding elements into a Collection

Once you create a collection, you get an empty collection. Now you need to add elements to it. When you add an element, two important steps happen in the background:

1. The information is vectorized (as configured in the collection definition)
2. The HNSW index is updated to optimize search (as you saw in the lectures). This occurs in the backend and you don't see it, but this can make the process take a bit of time

Adding elements is completed using a `collection.batch`, which adds additional useful features. For example, it will let you decide how many objects to send in each batch, handle errors during import, and improve performance by reducing the number of individual network calls. In this example, one element is added at a time, with only a single concurrent request at a time.

You can add a uuid (unique identifier id) to each element you add, so this prevents duplicate entries in your database.

Let's see in practice!

1. 插入数据时，后台会自动做两件事

1)把指定的文本字段转换成向量：

place + description + attractions + ...
        ↓
生成向量

2)更新 HNSW index：

新向量加入 HNSW 图结构
        ↓
以后可以更快地做相似度搜索



2. 通常使用 collection.batch 批量插入

批量插入的优点是：

减少网络请求次数
插入速度更快
可以控制每批插入多少条
更方便处理错误

3. UUID 用于唯一标识数据
UUID 是给每一条数据分配的唯一编号，类似身份证号。
例如：
{
    "uuid": "550e8400-e29b-41d4-a716-446655440000",
    "place": "Tokyo"
}
它的作用主要是区分不同数据，并避免重复插入。

In [ ]:
# 使用固定大小的 batch（批处理）方式向 collection 中插入数据
# batch_size=1：每次只发送 1 条数据
# concurrent_requests=1：同一时间只发送 1 个请求
with collection.batch.fixed_size(
    batch_size=1,
    concurrent_requests=1
) as batch:

    # 遍历 data 中的每一条数据
    # tqdm 会显示插入进度条
    for document in tqdm(data):

        # 根据 document 的内容生成一个固定的 UUID
        # 相同的 document 通常会生成相同的 UUID，
        # 从而帮助识别重复数据
        uuid = generate_uuid5(document)

        # 将当前 document 加入 batch
        batch.add_object(
            # properties 要求传入字典
            # document 中的 key 应与 collection 定义的字段名一致
            properties=document,

            # 为这条数据指定唯一标识符
            uuid=uuid,
        )

Awesome! Now you have a collection with vectors! You can check the number of vectors using `len(collection)`:

In [ ]:
len(collection)

<a id='3'></a>
## 3 - Querying on a collection

In this section, you will learn how to query on a collection. You can:

- Query on metadata
- Query with semantic search
- Query with BM25
- Query with filtering

Let's see some examples.

<a id='3-1'></a>
### 3.1 Filters

Before diving into querying, let's understand the Filters. Filters are a way of restricting your search on some criteria. They are very flexible. You usually pass them as an argument in a query. Let's have an example to illustrate it.

In [ ]:
# Here we are fetching 2 objects with a filter by property, filtering by 'user_ratings, only objects with value greater or equal to 3.5'
result = collection.query.fetch_objects(limit = 2, filters = Filter.by_property('user_ratings').greater_or_equal(3.5))

The result is an object called QueryReturn:

In [ ]:
result

You can access its objects by `result.objects`

In [ ]:
result.objects

So, each element in the list is an element of the collection.

In [ ]:
obj = result.objects[0]

You can check their properties, which is a dictionary.

In [ ]:
obj.properties

In this course, the way of filtering is `.by_property`. You will see more Filtering examples as other query methods are explained.

<a id='3-2'></a>
### 3.2 Semantic Search

You can use semantic search to query over your collection. This uses the vectors to compute distances between them and return the closest ones. You must pass a query, which will be vectorized and then compared over the elements on your collection. The method is `.near_text`.

In [ ]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.', limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print_object_properties(obj.properties)

You can also already query over the elements with `budget = Low`:

In [ ]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.',
                                    filters = Filter.by_property('budget').equal('Low'),
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print_object_properties(obj.properties)

You can also pass a list on the possible values on a filter, by using `.contains_any`:

In [ ]:
result = collection.query.near_text(query = 'I want suggestions to travel during Winter. I want cheap places.',
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
attractions: Broadway Theaters, New Year’s Eve Ball Drop
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Winter
state: New York
budget: Low
place: Times Square
user_ratings: 4.3

attractions: Going-to-the-Sun Road, Grinnell Glacier
description: Park known for its rugged mountains and alpine forests.
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Summer
state: Montana
budget: Moderate
place: Glacier National Park
user_ratings: 4.8

description: Beautiful park known for its impressive canyons and towering cliffs.
attractions: The Narrows, Angels Landing
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Spring, Fall
state: Utah
budget: Moderate
place: Zion National Park
user_ratings: 4.7

attractions: Nantucket Island, Cape Cod National Seashore
description: Popular tourist destination known for its beaches and quaint towns.
last_updated: 2023-10-01 00:00:00+00:00
best_season_t

<a id='3-3'></a>
### 3.3 BM25 search

To perform BM25 search, just run `colections.query.bm25`, the usual parameters `query`, `limit` and `filters` can be passed.

In [ ]:
result = collection.query.bm25(query = 'I want suggestions to travel during Winter. I want cheap places.',
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
attractions: Broadway Theaters, New Year’s Eve Ball Drop
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Winter
state: New York
budget: Low
place: Times Square
user_ratings: 4.3



<a id='3-4'></a>
### 3.4 Hybrid Search

This search is the RRF search you saw in the lectures. Apart from the standard parameters for querying, you can pass an `alpha` to control how much of BM25 you want in to mix in.

In [ ]:
result = collection.query.hybrid(query = 'I want suggestions to travel during Winter. I want cheap places.',
                                    filters = Filter.by_property('budget').contains_any(['Low','Moderate']),
                                    alpha = 0.3,
                                    limit = 4)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print_object_properties(obj.properties)

description: Bustling pedestrian intersection and major commercial hub.
attractions: Broadway Theaters, New Year’s Eve Ball Drop
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Winter
state: New York
budget: Low
place: Times Square
user_ratings: 4.3

description: Park known for its rugged mountains and alpine forests.
attractions: Going-to-the-Sun Road, Grinnell Glacier
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Summer
state: Montana
budget: Moderate
place: Glacier National Park
user_ratings: 4.8

description: Beautiful park known for its impressive canyons and towering cliffs.
attractions: The Narrows, Angels Landing
last_updated: 2023-10-01 00:00:00+00:00
best_season_to_visit: Spring, Fall
state: Utah
budget: Moderate
place: Zion National Park
user_ratings: 4.7

description: Popular tourist destination known for its beaches and quaint towns.
attractions: Nantucket Island, Cape Cod National Seashore
last_updated: 2023-10-01 00:00:00+00:00
best_season_t

<a id='3-5'></a>
### 3.5 Reranking

You can easily perform reranking with Weaviate by passing a new argument to a search. Let's try with semantic search!

In [ ]:
# 导入 Rerank 配置类
from weaviate.classes.query import Rerank

# 先使用 near_text 做向量语义检索，
# 再使用 reranker 对初步结果重新排序
response = collection.query.near_text(

    # 用户的原始查询：
    # 想在冬天旅游，并且希望地点便宜、好玩
    query="I want suggestions to travel during Winter. "
          "I want cheap and fun places.",

    # 最终最多返回 5 条结果
    limit=5,

    # 对初步检索结果进行二次排序
    rerank=Rerank(

        # reranker 只根据 attractions 字段的内容重新判断相关性
        prop="attractions",

        # reranker 使用的新查询
        # 它会重点判断哪些 attractions 更符合 “好玩的地方”
        query="Fun places"
    )
)

In [ ]:
# Let's iterate over the result objects and return their properties
for obj in result.objects:
    print_object_properties(obj.properties)

In [ ]:
# Don't forget to close the client!
client.close()

Keep it up! You just finished the basics of the Weaviate API you'll use in this course!